# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


Obervation: The above cell is importing all libraries along with obtaining all the requirements from readme.md. It also is creating paths for the folders in the file. 

## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

Observation: This cell is loading the csv files and is also creating a dictionary called Tables that groups the dataframes and mapping them with their corresponding names.

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


Oberservation: This cell is showing us the three rows of every file along with the columns. The time stamp in ratings might need to be fixed. Budget and revenue in movies is not in dollar values. 'tmdbId' is showing in different values from movies and links, it is showing in decimal instead of a whole number. 

In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


Obervation: This cell is providing a quick summary of each table or file from Tables. We can see how many rows each file has, how many columns, duplicate rows, and missing cells. The files contain the following numbers of rows, columns and missing cells: movies_metadata.csv 9082 rows, 21 columns, and 72 missing cells; ratings.csv 99810 rows, 5 columns, and no missing cells; links.csv 9125 rows, 3 columns, and 13 missing cells; credits_curated.csv 9082 rows, 3 columns, and 114 missing cells; and keywords_curated.csv 9082 rows, 2 columns, and 761 missing cells.

### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


Obervation: One observation is that here it is telling me how to combine the data. It is also showing no duplicate key or missing values.

## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question: How do genre, release year, runtime, director, cast, and keywords relate to audience ratings?
- Why this question matters: Research could provide insights into recommendations for their audiences which could improve user experience, and help understand which movie characteristics tend to be associate with higher or lower ratings. The analysis will look for patterns, not claim that any particular characteristic causes a movie to receive a higher or lower rating.
- Tables/files used: The project uses movies_metadata.csv, ratings.csv, links.csv, credits_curated.csv, and keywords_curated.csv. 
•	movies_metadata.csv contains title, original titles, original languages, release dates, runtime, budget, revenue, popularity, vote average, vote count, genres, production companies, production countries, rating, status, genre list, primary genre, and release years, movie lens, TMDB and IMDB ratings
•	ratings.csv contains ratings and rating date and times 
•	links.csv contains MovieLens movie IDs which can be used to map IMDb and TMDb IDs
•	credits_curated.csv contains director and cast information from TMDB
•	keywords_curated.csv contains movie keywords from TMDB
- Row meaning of each important table: Movies has movie-level data which provides the genre, release date, and runtime for each film along with rating from IMD and TMBD. Ratings contains ratings from MovieLens and movieid can be used to match the data. Links shows the movielens, IMDB, and TMDB mappings. The rows from credits is the data from TMDB which contains the directors, top cast, and keyword lists.
•	movies_metadata.csv (movies): has row that represents which movie and its metadata, such as genre, popularity, vote average and count, runtime, and release year
•	ratings.csv (ratings): has ratings from MovieLens, the userId and movieId that identify that rating event
•	links.csv (links): contains MovieLens movie’s ID mappings to external IMDB and TMDB
•	credits_curated.csv (credits): has the movie’s curated director and cast information for TMDB only
•	keywords_curated.csv (keywords): has the keywords associated with the movie for TMDB only
- Early data quality issues: The data  might be mismatched, inconsistent data types or formats, and any suspicous vlaues. The ratings might also be vary so it might skew the data, and lastly any older or niche movies might have less ratings or less data which can be seen as less favored but might not really be, they just might not have as many ratings. (we may need to account for shrinkage: a score based on only a handful of ratings doesn’t count as much as one based on many)
- Initial cleaning plan:  Save all data files separately first so the original files are preserved after changes are made, standardize all column names and data types so that they are matching across, parse release dates, and convert runtime and ratings to numeric values as well. Check for missing values, exact duplicate rows, duplicate IDs, and invalid values before changing or removing them. 
- Possible target/outcome: Broader understanding of the overall ratings bumped against each other for all movies. Analyze how each genre, release year, runtime, director, cast, and keywords (or other factors) are averaged and compare TMDb or IMDb ratings as additional outcomes only if the available files contain the ratings but ensure that I am taking into account the differences in rating scales and movie coverage.
- Ethics, privacy, or governance concerns: This data may not represent all viewers. Metadata may also be incomplete or uneven, so it might be biased data. I will also only keep the analysis at a movie level and not expose individual users' rating histories as that might raise some concerns.



In [6]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.



In [7]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



## Milestone 2 written workspace

### Business question
How are movie genre, release year, runtime, director, cast, and keywords associated with MovieLens audience ratings? This is a descriptive question about observed associations, not a claim that movie characteristics cause ratings. Movies with fewer than 20 MovieLens ratings are not assigned a high/low rating label because their averages are less stable.

### ABT unit and integration design
- **Unit of analysis:** one row per movie in the curated `movies_metadata.csv` catalog.
- **Primary key:** `movieId` (one unique MovieLens movie identifier per row). `tmdbId` is the integration key for curated credits and keywords, and is also checked for uniqueness in this catalog.
- **Base and source row counts:** `movies` has 9,082 rows; `ratings` has 99,810 rating events; `links` has 9,125 crosswalk rows; `credits` and `keywords` each have 9,082 rows.
- **Aggregated before joining:** `ratings` is grouped by `movieId` to compute mean, event count, standard deviation, first/last rating dates, and distinct user count. This converts the rating-event table to at most one row per movie. The aggregation covers 9,025 catalog movies; 57 catalog movies have no rating events.
- **Joins:** start with `movies`, then left join the rating summary on `movieId`, and left join `credits` and `keywords` on `tmdbId`. Each right-hand table is checked for key uniqueness and each merge uses a one-to-one validation. Left joins retain the full movie catalog. `links` is used as an ID crosswalk validation rather than joined again because `movies` already contains both IDs; all 9,082 catalog `movieId` values are present in `links`, with zero `tmdbId` mismatches.

### Transformations and validation
- Parse movie release dates and rating event dates as datetimes; retain the supplied identifier and descriptive fields without changing the source CSVs.
- For a movie with no rating events, set rating and distinct-user counts to 0; leave its mean, standard deviation, and first/last rating dates missing because those summaries are undefined.
- Derive `has_revenue` as an indicator for a positive recorded revenue. A zero is not proof that a film earned nothing; do not treat this field as a reliable revenue outcome or a pre-release predictor.
- Derive `rating_target_eligible` for movies with at least 20 rating events and `high_user_rating` as 1 when the eligible movie's mean MovieLens rating is at least 3.75 (on the 0.5-5 scale), otherwise 0. The target is missing for ineligible movies, not coded as 0. It is an observed post-release outcome, not a pre-release feature.
- Checks report row counts after every join, duplicate/missing keys, rating coverage, ABT missingness, and target coverage. The expected ABT has 9,082 rows, unique `movieId` keys, and no row multiplication.

### Remaining data quality concerns
The rating sample is limited to 671 users and a finite time period (1995-2016), so it may not represent all audiences. Fifty-seven catalog movies have no ratings; some other movies have very few ratings. `user_rating_std` is undefined for movies with only one rating. Curated cast, director, and keyword values have missing entries. `budget` and `revenue` contain zero values that may represent unavailable values rather than true zero amounts. Genres, production fields, cast, and keywords remain pipe-delimited or source-formatted text and would need additional parsing before feature-level modeling. Rating summaries and other post-release metadata must not be used as predictors for a pre-release decision.


In [8]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [9]:
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
ratings["rating_datetime"] = pd.to_datetime(ratings["rating_datetime"], errors="coerce")

assert movies["movieId"].notna().all() and movies["movieId"].is_unique
assert movies["tmdbId"].notna().all() and movies["tmdbId"].is_unique
assert credits["tmdbId"].notna().all() and credits["tmdbId"].is_unique
assert keywords["tmdbId"].notna().all() and keywords["tmdbId"].is_unique
assert not ratings[["userId", "movieId"]].duplicated().any()

rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max"),
        user_rating_distinct_users=("userId", "nunique"),
    )
    .reset_index()
)
assert rating_summary["movieId"].is_unique

join_checks = []
abt = movies.merge(
    rating_summary, on="movieId", how="left", validate="one_to_one"
)
join_checks.append({"join": "movies + rating summary on movieId", "rows_before": len(movies), "rows_after": len(abt), "unique_movieId": abt["movieId"].is_unique})

abt = abt.merge(credits, on="tmdbId", how="left", validate="one_to_one")
join_checks.append({"join": "+ credits on tmdbId", "rows_before": len(movies), "rows_after": len(abt), "unique_movieId": abt["movieId"].is_unique})

abt = abt.merge(keywords, on="tmdbId", how="left", validate="one_to_one")
join_checks.append({"join": "+ keywords on tmdbId", "rows_before": len(movies), "rows_after": len(abt), "unique_movieId": abt["movieId"].is_unique})

assert all(check["rows_after"] == len(movies) and check["unique_movieId"] for check in join_checks)
abt["user_rating_count"] = abt["user_rating_count"].fillna(0).astype("int64")
abt["user_rating_distinct_users"] = abt["user_rating_distinct_users"].fillna(0).astype("int64")
abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


Observation: I converted the release and rating timestamps to datetime, coercing invalid values to missing. I checked that movie IDs and TMDB IDs were present and unique where expected, and that each user–movie rating pair was unique. I grouped ratings by movie to calculate the mean, count, and standard deviation of ratings; the first and last rating dates; and the number of distinct users. I then left-joined these summaries with the movie, credit, and keyword data, validating the joins and confirming they preserved one row per movie. Finally, I filled missing rating counts and distinct-user counts with zero, created a revenue indicator, and derived rating eligibility and a high-rating label.

In [10]:
join_check_table = pd.DataFrame(join_checks)
key_checks = pd.DataFrame([
    {"table": "movies", "key": "movieId", "duplicate_keys": int(movies["movieId"].duplicated().sum()), "missing_key_values": int(movies["movieId"].isna().sum())},
    {"table": "movies", "key": "tmdbId", "duplicate_keys": int(movies["tmdbId"].duplicated().sum()), "missing_key_values": int(movies["tmdbId"].isna().sum())},
    {"table": "ratings", "key": "userId + movieId", "duplicate_keys": int(ratings[["userId", "movieId"]].duplicated().sum()), "missing_key_values": int(ratings[["userId", "movieId"]].isna().sum().sum())},
    {"table": "rating_summary", "key": "movieId", "duplicate_keys": int(rating_summary["movieId"].duplicated().sum()), "missing_key_values": int(rating_summary["movieId"].isna().sum())},
    {"table": "links", "key": "movieId", "duplicate_keys": int(links["movieId"].duplicated().sum()), "missing_key_values": int(links["movieId"].isna().sum())},
    {"table": "credits", "key": "tmdbId", "duplicate_keys": int(credits["tmdbId"].duplicated().sum()), "missing_key_values": int(credits["tmdbId"].isna().sum())},
    {"table": "keywords", "key": "tmdbId", "duplicate_keys": int(keywords["tmdbId"].duplicated().sum()), "missing_key_values": int(keywords["tmdbId"].isna().sum())},
])
assert (key_checks["duplicate_keys"] == 0).all()
assert (key_checks["missing_key_values"] == 0).all()

link_check = movies[["movieId", "tmdbId"]].merge(
    links[["movieId", "tmdbId"]], on="movieId", how="left",
    validate="one_to_one", suffixes=("_movies", "_links"), indicator=True,
)
link_check_summary = pd.Series({
    "catalog_movie_ids_missing_from_links": int(link_check["_merge"].eq("left_only").sum()),
    "tmdb_id_mismatches": int(link_check["tmdbId_movies"].ne(link_check["tmdbId_links"]).sum()),
})
assert (link_check_summary == 0).all()

abt_validation = pd.Series({
    "movies_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "abt_unique_movieId": int(abt["movieId"].nunique()),
    "duplicate_movieId_rows": int(abt["movieId"].duplicated().sum()),
    "movies_with_rating_events": int(abt["user_rating_count"].gt(0).sum()),
    "movies_without_rating_events": int(abt["user_rating_count"].eq(0).sum()),
    "movies_eligible_for_rating_target": int(abt["high_user_rating"].notna().sum()),
    "target_not_defined_rows": int(abt["high_user_rating"].isna().sum()),
})
assert len(abt) == len(movies) and abt["movieId"].is_unique

abt_missingness = pd.DataFrame({
    "missing_count": abt.isna().sum(),
    "missing_rate": abt.isna().mean(),
}).query("missing_count > 0").sort_values("missing_rate", ascending=False)

display(join_check_table)
display(key_checks)
display(link_check_summary)
display(abt_validation)
display(abt.head())
display(abt_missingness)


,join,rows_before,rows_after,unique_movieId
0,movies + rating summary on movieId,9082,9082,True
1,+ credits on tmdbId,9082,9082,True
2,+ keywords on tmdbId,9082,9082,True


,table,key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,rating_summary,movieId,0,0
4,links,movieId,0,0
5,credits,tmdbId,0,0
6,keywords,tmdbId,0,0


catalog_movie_ids_missing_from_links    0
tmdb_id_mismatches                      0
dtype: int64

movies_before_merges                 9082
abt_rows_after_merges                9082
abt_unique_movieId                   9082
duplicate_movieId_rows                  0
movies_with_rating_events            9025
movies_without_rating_events           57
movies_eligible_for_rating_target    1300
target_not_defined_rows              7782
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,user_rating_distinct_users,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,247,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,107,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,59,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,13,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,56,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_count,missing_rate
high_user_rating,7782,0.856860
user_rating_std,3099,0.341224
keywords_list,761,0.083792
top_cast,90,0.009910
last_rating_date,57,0.006276
user_rating_mean,57,0.006276
first_rating_date,57,0.006276
genres_list,35,0.003854
primary_genre,35,0.003854
director,24,0.002643


Observation: This section builds and displays validation summaries. It checks for duplicate or missing key values across the source and summary tables, verifies that movie IDs and TMDb IDs match between the movies and links tables, and confirms the merged ABT has one row per movie. It also reports rating coverage, rating-target eligibility, and missing values, and previews the completed ABT.

### SQL Validation Connected to the ABT

This SQLite query runs on the project `movies` and `ratings` DataFrames. It independently rebuilds the movie-level rating counts, means, and distinct-user counts in SQL, joins them to the movie catalog, and compares those results with the Pandas aggregation. SQLite runs in memory, so no account or database connection string is needed.

Expected interpretation: 9,082 rows and 9,082 distinct movie IDs means the SQL left join preserved the catalog grain. The SQL summary should cover 9,025 movies and all 99,810 rating events, with zero rating-summary mismatches against Pandas.


In [11]:
%pip install duckdb

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Observation: installing duckdb

In [12]:
import duckdb

connection = duckdb.connect(":memory:")
try:
    connection.register("movies", movies[["movieId"]])
    connection.register("ratings", ratings[["movieId", "userId", "rating"]])
    connection.register(
        "pandas_rating_summary",
        rating_summary[
            ["movieId", "user_rating_count", "user_rating_mean",
             "user_rating_distinct_users"]
        ],
    )
    
    sql_abt_check = connection.execute(
        """
        WITH sql_rating_summary AS (
                    SELECT
                        movieId,
                        COUNT(*) AS user_rating_count,
                        AVG(rating) AS user_rating_mean,
                        COUNT(DISTINCT userId) AS user_rating_distinct_users
                    FROM ratings
                    GROUP BY movieId
                )
                SELECT
                    COUNT(*) AS abt_rows,
                    COUNT(DISTINCT m.movieId) AS unique_movie_ids,
                    SUM(CASE WHEN s.movieId IS NOT NULL THEN 1 ELSE 0 END) AS movies_with_ratings,
                    SUM(COALESCE(s.user_rating_count, 0)) AS rating_events,
                    SUM(CASE WHEN
                        COALESCE(s.user_rating_count, 0) != COALESCE(p.user_rating_count, 0)
                        OR COALESCE(s.user_rating_distinct_users, 0) != COALESCE(p.user_rating_distinct_users, 0)
                        OR (s.user_rating_mean IS NULL AND p.user_rating_mean IS NOT NULL)
                        OR (s.user_rating_mean IS NOT NULL AND p.user_rating_mean IS NULL)
                        OR (s.user_rating_mean IS NOT NULL AND ABS(s.user_rating_mean - p.user_rating_mean) > 1e-10)
                    THEN 1 ELSE 0 END) AS rating_summary_mismatches
                FROM movies AS m
                LEFT JOIN sql_rating_summary AS s ON m.movieId = s.movieId
                LEFT JOIN pandas_rating_summary AS p ON m.movieId = p.movieId;
        """,
    ).df()
finally:
    connection.close()

assert sql_abt_check.loc[0, "abt_rows"] == len(movies)
assert sql_abt_check.loc[0, "unique_movie_ids"] == len(movies)
assert sql_abt_check.loc[0, "rating_summary_mismatches"] == 0
sql_abt_check


,abt_rows,unique_movie_ids,movies_with_ratings,rating_events,rating_summary_mismatches
0,9082,9082,9025.0,99810.0,0.0


Observation: I used DuckDB and recomputed each movie’s rating count, average rating, and number of distinct raters by grouping ratings on movieId. I left-joined those SQL results and the existing pandas rating summary to movies on movieId, then compared the summaries. I verified the joins preserved the movie row count and unique IDs and asserted that the rating summaries matched. This is a SQL validation of the rating summary, not the earlier joins with the credits and keywords tables.

### Preliminary ABT Data Dictionary

The dictionary below documents every retained ABT column. The notebook exports both the preliminary ABT and its matching data dictionary to `outputs/` for the project handoff.


In [13]:
dictionary_details = {
    "movieId": ("MovieLens movie identifier; primary key of the ABT.", "movies_metadata.csv", "Retained as the movie-level key.", "Must be unique and nonmissing."),
    "tmdbId": ("The Movie Database identifier for the movie.", "movies_metadata.csv", "Retained as the key for curated credits and keywords.", "Checked against links.csv; must be unique in the catalog."),
    "imdb_id": ("IMDb title identifier.", "movies_metadata.csv", "Retained without change.", "Identifier only; coverage may vary."),
    "title": ("Movie title.", "movies_metadata.csv", "Retained without change.", "Display text; titles may not be unique."),
    "original_title": ("Movie title in its original release context.", "movies_metadata.csv", "Retained without change.", "May differ from localized title."),
    "original_language": ("Original language code.", "movies_metadata.csv", "Retained without change.", "Categorical code."),
    "release_date": ("Movie release date.", "movies_metadata.csv", "Parsed as datetime; invalid values would become missing.", "Available values describe historical releases."),
    "runtime": ("Movie runtime in minutes.", "movies_metadata.csv", "Retained as numeric minutes.", "Check for implausible or missing values before modeling."),
    "budget": ("Reported production budget.", "movies_metadata.csv", "Retained as provided.", "Zero may indicate unavailable or unreported budget."),
    "revenue": ("Reported movie revenue.", "movies_metadata.csv", "Retained as provided.", "Post-release field; zero may be unavailable, not true zero revenue."),
    "popularity": ("Source popularity measure.", "movies_metadata.csv", "Retained as provided.", "Post-release measure; not a pre-release predictor."),
    "vote_average": ("Source-site average vote score.", "movies_metadata.csv", "Retained as provided.", "Different source and potentially different scale from MovieLens ratings; post-release."),
    "vote_count": ("Number of source-site votes.", "movies_metadata.csv", "Retained as provided.", "Post-release measure; not a pre-release predictor."),
    "genres": ("Source genre representation.", "movies_metadata.csv", "Retained as source-formatted text.", "Structured text; parse before feature-level use."),
    "production_companies": ("Source production company representation.", "movies_metadata.csv", "Retained as source-formatted text.", "Structured text; may contain multiple companies."),
    "production_countries": ("Source production country representation.", "movies_metadata.csv", "Retained as source-formatted text.", "Structured text; may contain multiple countries."),
    "adult": ("Source adult-content indicator.", "movies_metadata.csv", "Retained as provided.", "Check source semantics before interpretation."),
    "status": ("Source release/status label.", "movies_metadata.csv", "Retained as provided.", "Categorical field; may be missing or inconsistent."),
    "genres_list": ("Pipe-delimited movie genre names.", "movies_metadata.csv", "Retained as curated text.", "Multiple genres are represented in one field."),
    "primary_genre": ("Primary genre assigned to the movie.", "movies_metadata.csv", "Retained as provided.", "Missing for some movies; does not capture all genres."),
    "release_year": ("Calendar year of release.", "movies_metadata.csv", "Retained as provided.", "Derived in the curated source; confirm date/year consistency if needed."),
    "user_rating_mean": ("Mean MovieLens rating for this movie.", "ratings.csv", "Arithmetic mean of rating events grouped by movieId.", "Missing when there are no rating events; event averages are less stable for low counts."),
    "user_rating_count": ("Number of MovieLens rating events for this movie.", "ratings.csv", "Count of rating rows grouped by movieId; absent groups set to 0 after left join.", "A rating event count, not necessarily a count of distinct people."),
    "user_rating_std": ("Standard deviation of MovieLens ratings for this movie.", "ratings.csv", "Sample standard deviation grouped by movieId.", "Missing with no ratings or only one rating."),
    "first_rating_date": ("Date/time of the earliest MovieLens rating event.", "ratings.csv", "Minimum rating_datetime grouped by movieId.", "Missing when the movie has no rating events."),
    "last_rating_date": ("Date/time of the latest MovieLens rating event.", "ratings.csv", "Maximum rating_datetime grouped by movieId.", "Missing when the movie has no rating events."),
    "user_rating_distinct_users": ("Number of distinct MovieLens users rating this movie.", "ratings.csv", "Count distinct userId values grouped by movieId; absent groups set to 0.", "Users may rate more than one movie."),
    "director": ("Curated director name(s).", "credits_curated.csv", "Left joined on tmdbId.", "Missing values remain missing."),
    "top_cast": ("Curated top cast names.", "credits_curated.csv", "Left joined on tmdbId.", "Pipe-delimited text; may be missing."),
    "keywords_list": ("Curated movie keywords.", "keywords_curated.csv", "Left joined on tmdbId.", "Pipe-delimited text; may be missing."),
    "has_revenue": ("Indicator that a positive revenue value is recorded.", "movies_metadata.csv", "1 when numeric revenue is greater than 0; otherwise 0.", "Zero is not proof of no revenue; do not use as a pre-release predictor."),
    "rating_target_eligible": ("Whether the movie has enough rating events for the target definition.", "ratings.csv", "True when user_rating_count is at least 20.", "Eligibility rule is a project choice, not a general standard."),
    "high_user_rating": ("High MovieLens audience rating label for eligible movies.", "ratings.csv", "1 when eligible mean rating is at least 3.75; 0 otherwise; missing when ineligible.", "Observed post-release outcome; not a pre-release predictor."),
}
assert set(abt.columns) == set(dictionary_details)

data_dictionary = pd.DataFrame([
    {
        "column_name": column,
        "data_type": str(abt[column].dtype),
        "description": dictionary_details[column][0],
        "source": dictionary_details[column][1],
        "transformation_or_derivation": dictionary_details[column][2],
        "notes_or_concerns": dictionary_details[column][3],
    }
    for column in abt.columns
])

abt_path = OUTPUT_DIR / "movies_preliminary_abt.csv"
dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
abt.to_csv(abt_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)
print(f"Saved {len(abt):,} ABT rows to {abt_path.relative_to(REPO_ROOT)}")
print(f"Saved {len(data_dictionary):,} dictionary entries to {dictionary_path.relative_to(REPO_ROOT)}")
data_dictionary


Saved 9,082 ABT rows to outputs/movies_preliminary_abt.csv
Saved 33 dictionary entries to outputs/movies_data_dictionary.csv


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,MovieLens movie identifier; primary key of the...,movies_metadata.csv,Retained as the movie-level key.,Must be unique and nonmissing.
1,tmdbId,int64,The Movie Database identifier for the movie.,movies_metadata.csv,Retained as the key for curated credits and ke...,Checked against links.csv; must be unique in t...
2,imdb_id,str,IMDb title identifier.,movies_metadata.csv,Retained without change.,Identifier only; coverage may vary.
3,title,str,Movie title.,movies_metadata.csv,Retained without change.,Display text; titles may not be unique.
4,original_title,str,Movie title in its original release context.,movies_metadata.csv,Retained without change.,May differ from localized title.
5,original_language,str,Original language code.,movies_metadata.csv,Retained without change.,Categorical code.
6,release_date,datetime64[us],Movie release date.,movies_metadata.csv,Parsed as datetime; invalid values would becom...,Available values describe historical releases.
7,runtime,float64,Movie runtime in minutes.,movies_metadata.csv,Retained as numeric minutes.,Check for implausible or missing values before...
8,budget,int64,Reported production budget.,movies_metadata.csv,Retained as provided.,Zero may indicate unavailable or unreported bu...
9,revenue,float64,Reported movie revenue.,movies_metadata.csv,Retained as provided.,"Post-release field; zero may be unavailable, n..."


Observation: This section documents the preliminary movie-level ABT and its columns in a data dictionary, including their data types, sources, transformations, and caveats. It verifies that every ABT column has a dictionary entry, then saves the ABT and dictionary as CSV files and reports the ABT row count. Key uniqueness and missingness are validated in an previous section.

AI Disclosure: Used AI to decode and debug: I could not figure out the SQL DuckDB portion as I was getting errors, so I used AI to help me walk through the errors. Also used to assist with Technical Reference I didn’t understand when using DuckDB. AI was also used to ensure grammar was correct for the observations. 

## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [14]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [15]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,NaN,NaN,NaN,1992-06-23 04:19:23.796520,1902-09-01 00:00:00,1984-12-21 00:00:00,1997-09-01 00:00:00,2006-06-27 00:00:00,2016-09-23 00:00:00,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [16]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [17]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
